#Parameters

In [ ]:
release='release_tx842'
run_name='hypergeometric_relaxed'
overwrite=False # whether to overwrite existing results or not

# import functions

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import os, sys
from loguru import logger
from scipy import stats

try:
    logger.level("TEST", no=15, color="<magenta>", icon="🧪")
    logger.level("PROD", no=25, color="<blue>", icon="🌋")
except ValueError:
    pass  # Level already exists



def set_logging_mode(mode="TEST"):
    mode = mode.upper()
    if mode == "SILENT":
        return
    if mode not in ["TEST", "PROD"]:
        raise ValueError("Mode must be 'TEST' or 'PROD'")
        
    logger.remove()  # Clear current configuration
    logger.add(sys.stdout, level=mode)
    print(f"🔄 Logging mode switched to: {mode}")


def explode_row(df, col_name):
    new_rows = []
    for _, row in df.iterrows():
        vals = row[col_name]
        if pd.isna(vals):
            new_rows.append(row)
        else:
            split_vals = vals.split(',')
            for new_val in split_vals:
                new_row = row.copy()
                new_row[col_name] = new_val
                new_rows.append(new_row)
    return pd.DataFrame(new_rows, columns=df.columns)


def get_qc_df(QC_table_path):
    qc_df = pd.read_csv(QC_table_path, sep="\t", low_memory=False)
    qc_df = qc_df[qc_df["selected_solution"]]
    qc_df["patient_tumour"] = (
        qc_df["patient"] + "-Tumour" + qc_df["tumour_id"].astype(int).astype(str)
    )
    qc_df_finalised_tumours = qc_df.groupby("patient_tumour").filter(
        lambda x: x["FINALIZED"].all()
    )
    return qc_df, qc_df_finalised_tumours


def get_alpaca_wgd_results(alpaca_output_dir):
    all_tumours = os.listdir(alpaca_output_dir)
    all_dfs = []
    for tumour_id in all_tumours:
        tumour_subdir = os.path.join(alpaca_output_dir, tumour_id)
        wgd_results_path = os.path.join(tumour_subdir, "wgd_ratio_scores.csv")
        if os.path.exists(wgd_results_path):
            all_dfs.append(pd.read_csv(wgd_results_path))
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)


def get_alpaca_results(alpaca_output_dir):
    combined_df = os.path.join(alpaca_output_dir.replace('tumour_results', 'cohort_results'), 'all_tumours_combined.csv')
    if os.path.exists(combined_df):
        return pd.read_csv(combined_df)
    else:
        print(f"Combined results file not found at {combined_df}")
        return None    
    

def load_clone_proportions_for_tumour(selected_tumour, alpaca_input_dir):
    clone_proportions_path = os.path.join(alpaca_input_dir, selected_tumour, "cp_table.csv")
    if os.path.exists(clone_proportions_path):
        df = pd.read_csv(clone_proportions_path, index_col='clone')
        df.columns = [col.replace('-', '.') for col in df.columns]
        return df
    else:
        print(f"Clone proportions file not found for tumour {selected_tumour}")
        return None
    

def get_ancestors(clone, alpaca_wgd_in_selected_tumour):
    ancestors = []
    while clone is not None:
        parent = alpaca_wgd_in_selected_tumour.query('clone == @clone').parent.iloc[0] if not alpaca_wgd_in_selected_tumour.query('clone == @clone').empty else None
        if parent is not None:
            ancestors.append(parent)
        clone = parent
    return ancestors


def get_children(clone, alpaca_wgd_in_selected_tumour):
    children = alpaca_wgd_in_selected_tumour.query('parent == @clone').clone.tolist() if not alpaca_wgd_in_selected_tumour.query('parent == @clone').empty else []
    return children


def get_descendants(clone, alpaca_wgd_in_selected_tumour):
    descendants = []
    children = get_children(clone, alpaca_wgd_in_selected_tumour)
    while children:
        this_child = children.pop(0)
        descendants.append(this_child)
        children.extend(get_children(this_child, alpaca_wgd_in_selected_tumour))
    return descendants


def additional_ccf_qc(qc_df_finalised_tumours, ccf_thr=0.9, release_path=None):
    all_dfs = []
    for patient_tumour in qc_df_finalised_tumours.patient_tumour.unique():
        patient_id = patient_tumour.split('-')[0]
        tumour_number = patient_tumour.split('-')[1].replace('Tumour', '')
        df_path = f'{release_path}/{patient_id}/mutation_trees/tumour_{tumour_number}/clusterInfo.txt'
        if os.path.exists(df_path):
            df=pd.read_csv(df_path, sep='\t')
            df = df.dropna(subset=['treeClust', 'truncal'])
            df=df[df.treeClust]
            df_trunk = df[df.truncal]
            df_trunk['tumour_id'] = patient_tumour
            df_trunk = df_trunk.sort_values(by='meanCCF', ascending=True)
            all_dfs.append(df_trunk.iloc[0])
    df_all_trunk = pd.DataFrame(all_dfs)
    df_all_trunk = df_all_trunk[['tumour_id', 'clusterID', 'meanCCF']]
    df_all_trunk.meanCCF.plot.hist(title='Distribution of minimum mean CCF for clonal clusters across all tumours with QC=Finalised')
    return df_all_trunk, qc_df_finalised_tumours[qc_df_finalised_tumours.patient_tumour.isin(df_all_trunk[df_all_trunk.meanCCF > ccf_thr].tumour_id.tolist())]


def fisher_explain(gd_event_id, mut_supported_df, clone=None, print_output=False):
    if clone is not None:
        mut_supported_df = mut_supported_df[mut_supported_df['cluster_id'] == clone]
    row = mut_supported_df[mut_supported_df['GD_event_id'] == gd_event_id].iloc[0]
    tumour_id = row['tumour_id']
    cluster_id = row['cluster_id']
    samples_supporting = row['samples']
    if print_output:
        print(row)
    k = row['num_cn2']
    K = row['num_cn2_all']
    n = row['num_muts_present']
    N = row['num_total_present']
    eligible_at_cn2 = k
    eligibile_at_other_cn = n - k
    non_eligible_at_cn2 = K-k
    non_eligible_at_lower_cn = N - K - n + k
    contingency_table = np.array([[eligible_at_cn2, eligibile_at_other_cn], [non_eligible_at_cn2, non_eligible_at_lower_cn]])   
    contingency_df = pd.DataFrame(contingency_table, columns=['CN=2', 'CN<2'], index=['Eligible', 'Non-eligible'])
    # add margins to contingency table:
    contingency_df.loc['Total'] = contingency_df.sum()
    contingency_df['Total'] = contingency_df.sum(axis=1)
    if print_output:
        print("Contingency Table:")
        print(contingency_df)
        print(f"Hypergeometric test p-value: {row['gd_p_value']}")
        print(f"GD Event ID: {gd_event_id}")
        print(f"Tumour ID: {tumour_id}")
        print(f"Cluster ID: {cluster_id}")
        print(f"Samples supporting GD: {samples_supporting}")
    return contingency_df


def order_clones_by_phylogeny(clones,tree_df):
    sub_tree_df = tree_df[tree_df.clone.isin(clones)]
    terminal_nodes = sub_tree_df[~sub_tree_df.clone.isin(sub_tree_df.parent)].clone
    ordered_lineages = []
    for terminal_node in terminal_nodes:
        ordered_clones = [terminal_node]
        while len(ordered_clones) < len(clones):
            current_clone = ordered_clones[-1]
            parent_clone = sub_tree_df[sub_tree_df.clone == current_clone].parent.iloc[0]
            if parent_clone in clones and parent_clone not in ordered_clones:
                ordered_clones.append(parent_clone)
            else:
                break
        ordered_lineages.append(ordered_clones)
    return ordered_lineages


def which_q(tumour_id):
    queue_files = '/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/queue_files'
    for file in os.listdir(queue_files):
        if file.endswith('queue.tsv'):
            df = pd.read_csv(os.path.join(queue_files, file), sep='\t')
            if tumour_id in df['tumour_id'].values:
                return file
    return None


# Load data

In [ ]:
results_dir = f'/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/wes/{release}/pgdd_output/{run_name}'
alpaca_output_dir = f'/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/tracerx800/output/tx842/snp_filt_0_cn_rec_FFT/tumour_results'
alpaca_input_dir = f'/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/tracerx800/input/tx842/snp_filt_0_cn_rec_FFT/'
events_path = f'{results_dir}/pgdd_GD_events.tsv'
per_region_path = f'{results_dir}/pgdd_GDs_per_region.tsv'
per_tumour_path = f'{results_dir}/pgdd_GDs_per_tumour.tsv'
mut_count_path = f'{results_dir}/pgdd_mut_counts.tsv'
summary_path = f'{results_dir}/pgdd_cohort_gd_summary.tsv'
pgdd_input_path = f'/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/wes/{release}/pgdd_input/{run_name}/pgdd_input_all_tumours.tsv'
mut_count_df = pd.read_csv(mut_count_path, sep='\t')
events_df = pd.read_csv(events_path, sep='\t')
per_region_df = pd.read_csv(per_region_path, sep='\t')
per_tumour_df = pd.read_csv(per_tumour_path, sep='\t')
pgdd_input_df = pd.read_csv(pgdd_input_path, sep='\t')
summary_df = pd.read_csv(summary_path, sep='\t')
# from the mut_count_df table, keep only rows where num_cn2 is non-zero
mut_count_df_non_zero_cn2 = mut_count_df[mut_count_df['num_cn2'] > 0]

# load pipeline results for comparison:
pipeline_results_dir = f'/nemo/project/proj-tracerx-lung/tracerx/_RELEASE/release_tx842/_aggregate/pgddetect'
pipeline_events_path = f'{pipeline_results_dir}/20250809_cohort_GDs_events.tsv'
pipeline_per_region_path = f'{pipeline_results_dir}/20250809_cohort_GDs_per_region.tsv'
pipeline_per_tumour_path = f'{pipeline_results_dir}/20250809_cohort_GDs_per_tumour.tsv'
pipeline_summary_path = f'{pipeline_results_dir}/20250809_cohort_gd_summary.tsv'

pipeline_events_df = pd.read_csv(pipeline_events_path, sep='\t')
pipeline_per_region_df = pd.read_csv(pipeline_per_region_path, sep='\t')
pipeline_per_tumour_df = pd.read_csv(pipeline_per_tumour_path, sep='\t')
pipeline_summary_df = pd.read_csv(pipeline_summary_path, sep='\t')


# sanitise names:
pipeline_events_df['tumour_id'] = pipeline_events_df['tumour_id'].str.replace('_tumour', '-Tumour')
pipeline_per_region_df['tumour_id'] = pipeline_per_region_df['tumour_id'].str.replace('_tumour', '-Tumour')
pipeline_per_tumour_df['tumour_id'] = pipeline_per_tumour_df['tumour_id'].str.replace('_tumour', '-Tumour')

tumour_count_before_qc = per_tumour_df.tumour_id.nunique()
# add the QC table to ensure we are not chasing non-qc-ed tumours:
QC_table_path = "/nemo/project/proj-tracerx-lung/tracerx/_RELEASE/release_tx842/_aggregate/20250729_tx842_manual_qc_complete_post_refphase.tsv"
qc_df, qc_df_finalised_tumours = get_qc_df(QC_table_path)
tumour_count_where_FINALISED_eq_TRUE = qc_df_finalised_tumours['patient_tumour'].nunique()

# additional QC: remove tumours where CCF values are not ~100 in the clonal cluster in all regions:

# subset all dataframes to keep only fully QC-ed tumours:
df_all_trunk, qc_df_finalised_tumours_ccf_qced = additional_ccf_qc(qc_df_finalised_tumours,ccf_thr=0.9, release_path='/nemo/project/proj-tracerx-lung/tracerx/_RELEASE/release_tx842')
tumour_count_after_additional_ccf_qc = qc_df_finalised_tumours_ccf_qced['patient_tumour'].nunique()

per_region_df = per_region_df[per_region_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
per_tumour_df = per_tumour_df[per_tumour_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
events_df = events_df[events_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
mut_count_df = mut_count_df[mut_count_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
mut_count_df_non_zero_cn2 = mut_count_df_non_zero_cn2[mut_count_df_non_zero_cn2.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
pipeline_events_df = pipeline_events_df[pipeline_events_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
pipeline_per_region_df = pipeline_per_region_df[pipeline_per_region_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
pipeline_per_tumour_df = pipeline_per_tumour_df[pipeline_per_tumour_df.tumour_id.isin(qc_df_finalised_tumours_ccf_qced["patient_tumour"])]
summary_df = summary_df[summary_df.sample_name_hash.isin(per_region_df.sample_id)]
summary_df.rename(columns={'sample_name_hash': 'sample_id'}, inplace=True)

tumour_count_after_qc = per_tumour_df.tumour_id.nunique()


In [ ]:

# for exploded events:
events_df_exploded = explode_row(events_df, 'clusters')
events_df_exploded['clusters'] = events_df_exploded['clusters'].str.split('clone').str[1]


In [ ]:
events_df_exploded['clusters'] = events_df_exploded['clusters'].fillna('-1')
events_df_exploded['clusters'] = events_df_exploded['clusters'].astype(int)

In [ ]:

#events_df_exploded['clusters'] = events_df_exploded['clusters'].astype(int)
events_df_exploded.rename(columns={'clusters': 'cluster_id'}, inplace=True)
events_df_exploded_mut_count = events_df_exploded.merge(mut_count_df, on=['tumour_id', 'cluster_id'], how='left')
mut_supported = events_df_exploded_mut_count.sort_values('gd_p_value').dropna(subset=['gd_p_value'])
mut_supported['significant'] = mut_supported['gd_p_value'] < 0.05
mut_supported['power_proxy (n/N)'] = mut_supported.num_muts_present/mut_supported.num_total_present

In [ ]:
print(f"Number of tumours before QC: {tumour_count_before_qc}")
print(f"Number of tumours where FINALISED eq TRUE: {tumour_count_where_FINALISED_eq_TRUE}")
print(f"Number of tumours after additional CCF QC: {tumour_count_after_additional_ccf_qc}")
print(f"Number of tumours after QC: {tumour_count_after_qc}")

In [ ]:
# show examples of tumours with ccf below threshold:
tumours_below_ccf_threshold = qc_df_finalised_tumours.patient_tumour[~qc_df_finalised_tumours.patient_tumour.isin(qc_df_finalised_tumours_ccf_qced.patient_tumour)]

In [ ]:
# Assign Alpaca calls to events_df table:
ALPACA_WGD_threshold = 1.5
alpaca_wgd_results_df = get_alpaca_wgd_results(alpaca_output_dir)
# split events_per_tumour into categories:
per_tumour_df_no_gd = per_tumour_df[per_tumour_df['First_GD'] == 'No GD']
per_tumour_df_clonal_gd_only = per_tumour_df[(per_tumour_df['First_GD'] == 'Clonal') & (per_tumour_df['num_subclonal_gds'] == 0)]

gd_to_process = per_tumour_df[~per_tumour_df.tumour_id.isin(per_tumour_df_no_gd.tumour_id) & ~per_tumour_df.tumour_id.isin(per_tumour_df_clonal_gd_only.tumour_id)]


# create resolved queue files for no gd and clonal gd cases:
queue_files = '/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/queue_files'

per_tumour_df_no_gd_queue = per_tumour_df_no_gd[['tumour_id', 'First_GD']]
per_tumour_df_no_gd_queue['status'] = 'resolved'
per_tumour_df_no_gd_queue['memo'] = 'No genome doubling detected'
if overwrite or not os.path.exists(f'{queue_files}/pgdd_wes_no_gd_queue.tsv'):
    per_tumour_df_no_gd_queue.to_csv(f'{queue_files}/pgdd_wes_no_gd_queue.tsv', sep='\t', index=False)

per_tumour_df_clonal_gd_only_queue = per_tumour_df_clonal_gd_only[['tumour_id', 'First_GD']]
per_tumour_df_clonal_gd_only_queue['status'] = 'resolved'
per_tumour_df_clonal_gd_only_queue['memo'] = 'Clonal genome doubling detected, no subclonal genome doubling detected'
if overwrite or not os.path.exists(f'{queue_files}/pgdd_wes_clonal_gd_only_queue.tsv'):
    per_tumour_df_clonal_gd_only_queue.to_csv(f'{queue_files}/pgdd_wes_clonal_gd_only_queue.tsv', sep='\t', index=False)

This leaves us with 143 tumours which have some number of subclonal events and some number of clonal events:

Next, split remaining tumours into the ones where we have any evidence from mutations and those where we don't have such evidence:

In [ ]:
has_mut_supported_calls = events_df.groupby('tumour_id').apply(lambda x: x.is_subclonal_mutation_supported.any())
has_mut_supported_calls = has_mut_supported_calls[has_mut_supported_calls].index.tolist()
gd_to_process_no_mut_support = gd_to_process[~gd_to_process.tumour_id.isin(has_mut_supported_calls)]
gd_to_process_mut_support = gd_to_process[gd_to_process.tumour_id.isin(has_mut_supported_calls)]

# resolve ploidy based
# start with cases with one clonal and some subclonal GD, they all should have easy asignment to clones:
clonal_and_subclonal_gd = gd_to_process_no_mut_support[(gd_to_process_no_mut_support['First_GD'] == 'Clonal') & (gd_to_process_no_mut_support['num_subclonal_gds'] > 0)]

# this is for post-hoc manual verification. If ratio is not convincing, check in wgd_viewer and reconcile.
# TODO assign 'problematic' tag to events with low ratio_A values, and check in wgd_viewer for confirmation.
events_df['alpaca_ratio_A'] = np.nan

# iterate over tumours and codify the algorithm
for selected_tumour in gd_to_process_no_mut_support.tumour_id:#clonal_and_subclonal_gd.tumour_id:
    #if selected_tumour != 'LTX0755-Tumour2':
    #    continue
    # check subclonal events called by PGDD:
    pgdd_events_in_selected_tumour = events_df.query('tumour_id == @selected_tumour').query('is_clonal == False')
    clone_proportions_in_selected_tumour = load_clone_proportions_for_tumour(selected_tumour, alpaca_input_dir)
    alpaca_wgd_in_selected_tumour = alpaca_wgd_results_df.query('tumour_id == @selected_tumour').query('parent != "diploid"').sort_values(by='ratio_A', ascending=False)
    # for each event, find best ALPACA match:
    alpaca_events = {}
    for subclonal_event in pgdd_events_in_selected_tumour.itertuples():
        # check each ALPACA candidate. Load clone proportions to ensure that alpaca events match the samples from PGDD.
        samples_from_pgdd_event = subclonal_event.samples.split(',')
        cp_samples = clone_proportions_in_selected_tumour[samples_from_pgdd_event]
        clones_present = cp_samples.sum(axis=1)
        clones_present = clones_present[clones_present > 0].index.tolist()
        # add all the extinct ancestors to the search space:
        all_ancestors = sum([get_ancestors(clone, alpaca_wgd_in_selected_tumour) for clone in clones_present],[])
        clones_present = list(set(clones_present + all_ancestors))
        # pick top (highest ratio_A) ALPACA event that matches the clones present in the PGDD event:
        alpaca_wgd_in_selected_tumour_selected_clones = alpaca_wgd_in_selected_tumour[alpaca_wgd_in_selected_tumour.clone.isin(clones_present)]
        # filter clones alredy seleted:
        alpaca_wgd_in_selected_tumour_selected_clones = alpaca_wgd_in_selected_tumour_selected_clones[~alpaca_wgd_in_selected_tumour_selected_clones.clone.isin(alpaca_events.values())]
        alpaca_events[subclonal_event.GD_event_id] = alpaca_wgd_in_selected_tumour_selected_clones.clone.iloc[0] if not alpaca_wgd_in_selected_tumour_selected_clones.empty else None
        # write back to events_df:
        events_df.loc[events_df.GD_event_id == subclonal_event.GD_event_id, 'clusters'] = 'alpaca_' + str(alpaca_events[subclonal_event.GD_event_id])
        events_df.loc[events_df.GD_event_id == subclonal_event.GD_event_id, 'alpaca_ratio_A'] = alpaca_wgd_in_selected_tumour_selected_clones.ratio_A.iloc[0] if not alpaca_wgd_in_selected_tumour_selected_clones.empty else np.nan


In [ ]:
# create a queue file for no mutation support cases:
events_df_alpaca_called = events_df[(events_df['clusters'].notna()) & (events_df['clusters'].str.startswith('alpaca_'))]

records = []
for tumour_id in events_df_alpaca_called.tumour_id.unique():
    tumour_events = events_df_alpaca_called[events_df_alpaca_called.tumour_id == tumour_id]
    events_ratios = tumour_events.alpaca_ratio_A.tolist()
    all_pass = all(ratio > ALPACA_WGD_threshold for ratio in events_ratios)
    status = 'resolved' if all_pass else 'unresolved'
    memo = 'All events have ratio_A > threshold' if all_pass else 'Some events have ratio_A < threshold, check in wgd_viewer'
    record = {
        'tumour_id': tumour_id,
        'status': status,
        'memo': memo
    }
    records.append(record)
resolved_queue_df = pd.DataFrame(records)
if overwrite or not os.path.exists(f'{queue_files}/pgdd_wes_alpaca_called_queue.tsv'):
    resolved_queue_df.to_csv(f'{queue_files}/pgdd_wes_alpaca_called_queue.tsv', sep='\t', index=False)

In [ ]:
# create first version of mutation supported queue file:
records = []
for tumour_id in gd_to_process_mut_support.tumour_id.unique():
    tumour_events = events_df[events_df.tumour_id == tumour_id]
    status = 'unresolved'
    tumour_events_string = 'PGDD tumour events:/n' + '\n'.join(tumour_events.GD_event_id.tolist()) + '\n'
    try:
        contingency_tables = [fisher_explain(gd_event_id, mut_supported) for gd_event_id in tumour_events.GD_event_id.tolist()]
        contingency_tables_string = '\n'.join([str(table) for table in contingency_tables])
        memo = f'Tumour events:\n{tumour_events}\nContingency tables:\n{contingency_tables_string}'
    except IndexError:
        memo = ''
    record = {
        'tumour_id': tumour_id,
        'status': status,
        'memo': memo
    }
    records.append(record)
mut_supported_queue_df = pd.DataFrame(records)
if overwrite or not os.path.exists(f'{queue_files}/pgdd_wes_mut_supported_queue.tsv'):
    mut_supported_queue_df.to_csv(f'{queue_files}/pgdd_wes_mut_supported_queue.tsv', sep='\t', index=False)

# process cases with mutation support

In [ ]:
events_df_mut_supported = events_df[events_df.tumour_id.isin(gd_to_process_mut_support.tumour_id)]
# check how many of the mut_supported were previously flagged as problematic:
problematic_df = pd.read_csv('/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/_assets/prototype_events.tsv', sep='\t')
problematic_df = problematic_df[problematic_df.tumour_id.isin(events_df_mut_supported.tumour_id)]

In [ ]:
reviewed_tumours = ['LTX0357-Tumour1','LTX0965-Tumour1', 'LTX0387-Tumour1', 'LTX1020-Tumour1' ,'LTX0997-Tumour1', 'LTX0272-Tumour1', 'LTX1261-Tumour1', 'LTX0944-Tumour1', 'LTX1238-Tumour1', 'LTX0304-Tumour1', 'LTX0929-Tumour1','LTX1190-Tumour1']
problematic_df[~problematic_df.tumour_id.isin(reviewed_tumours)].sort_values('tumour_max_problematic_level', ascending=False).query("tumour_max_problematic_level > 0")

In [ ]:
# now spot-check a few non-problematic ones to ensure they are indeed non-problematic:
problematic_df[~problematic_df.tumour_id.isin(reviewed_tumours)].sort_values('tumour_max_problematic_level', ascending=False).query("tumour_max_problematic_level == 0").tumour_id.unique()

In [ ]:
# apply the same method as in 'resolve ploidy based' to the ploidy based calls, with the difference that here the low ratio signifies that we are not sure about the exact timing of the event, not its existence (becasuse this is supported by the mutations in other clones).



Add algorithm to time the GDs withing the ploidy supported section. For example in LTX0019-Tumour1 it should probably call clone3, in LTX0064-Tumour1 probably clone 32 and clone 1.

Ideas to higlight problematic ones:
LTX0073: mutation support (40 muts) in cluster 4.

#here

In [ ]:
# resolve PGDD multi cluster cases. If both clusters are in direct parent-child relationship, move the event to the parent.
# Otherwise, flag:
events_df_multicluster = events_df[events_df.clusters.astype(str).str.contains(',')]
for event in events_df_multicluster.itertuples():
    clusters = [f'clone{cluster}' for cluster in event.clusters.split(',')]
    if len(clusters) != 2:
        logger.warning(f"Event {event.GD_event_id} in tumour {event.tumour_id} has more than two clusters: {clusters}. Skipping.")
        continue
    clusters_set = set(clusters)
    tumour_id = event.tumour_id
    tree_df = alpaca_wgd_results_df.query('tumour_id == @tumour_id')[['clone', 'parent']]
    tree_df_tuples = [ set(x) for x in list(tree_df.itertuples(index=False, name=None)) ]
    if clusters_set in tree_df_tuples:
        tree_df = tree_df[tree_df.parent.isin(clusters) & tree_df.clone.isin(clusters)]
        child = tree_df.clone.iloc[0]
        events_df.loc[events_df.GD_event_id == event.GD_event_id, 'clusters'] = 'multicluster_' + child         

In [ ]:
# resolve clonal cases:
for event in events_df.itertuples():
    tumour_id = event.tumour_id
    clonal = event.is_clonal
    if clonal:
        tree_df = alpaca_wgd_results_df.query('tumour_id == @tumour_id')[['clone', 'parent']]
        mrca = tree_df[tree_df.parent == 'diploid'].clone.iloc[0]
        events_df.loc[events_df.GD_event_id == event.GD_event_id, 'clusters'] = 'clonal_' + mrca

In [ ]:
# for each genome doubling event called on the ploidy level, find the most likely clone of origin.
# Stage 1: split cases into easy and hard. 
# easy cases are the ones where entire doubled sample is composed of private clones and none of these clones is in the lineage of any other clone with mutation support.

# TODO clean this up later, for now we append '_resolved' to each new copy to avoid breaking mutations
events_df_resolved = events_df.copy()

# We are only considering tumours with ploidy-based GD calls:
ploidy_supported_gd_events = events_df[(events_df.is_clonal == False) & (events_df.clusters.isna())]
testing_tumours = []
all_tumours = ploidy_supported_gd_events.tumour_id.unique()
if testing_tumours:
    all_tumours = testing_tumours

for tumour_id in all_tumours:
    print('-'*30)
    print(f"Processing tumour {tumour_id}...")
    second_order_gds = (per_tumour_df.query('tumour_id == @tumour_id').num_second_gd>0).any()
    if second_order_gds:
        # TODO not sure how to handle this yet, so alert for now.
        print(f"Tumour {tumour_id} has second order GDs, cannot resolve placement of first order GDs without further information.")
        continue
    ploidy_supported_gd_events_in_selected_tumour = ploidy_supported_gd_events[ploidy_supported_gd_events.tumour_id == tumour_id]
    if ploidy_supported_gd_events_in_selected_tumour.empty:
        logger.log("TEST", f"No ploidy supported GD events found in tumour {tumour_id}, skipping.")
        continue
    for event_id in ploidy_supported_gd_events_in_selected_tumour.GD_event_id.unique():
        # check if event has been already resolved with ALPACA ratio calls:
        if 'alpaca_' in str(events_df.loc[events_df.GD_event_id == event_id, 'clusters'].iloc[0]):
            print(f"Event {event_id} in tumour {tumour_id} has already been resolved with ALPACA ratio calls, skipping.")
            continue
        print(f"Processing GD event {event_id} in tumour {tumour_id}...")
        event_subdf = ploidy_supported_gd_events[ploidy_supported_gd_events.GD_event_id == event_id]
        other_events = events_df[(events_df.tumour_id == tumour_id) & (events_df.GD_event_id != event_id)]
        # create a list of clones where this GD event cannot be place because of the conflict with the mutation based GD events:
        # TODO ensure this works properly if other GD events are ploidy based:
        forbidden_descendants = []
        if not other_events.empty:
            if other_events.clusters.isna().any():
                print(f"Other GD events in tumour {tumour_id} are ploidy based, cannot resolve placement of event {event_id} without further information.")
                continue
            elif other_events.clusters.astype(str).str.contains(',').any():
                print(f"Other GD events in tumour {tumour_id} are multi-clone, cannot resolve placement of event {event_id} without further information.")
                continue
            else:
                clones_in_other_events = [f'clone{c}' if 'clone' not in str(c) else str(c) for c in other_events.clusters.astype(str).tolist()]
                forbidden_descendants = sum([get_descendants(clone, alpaca_wgd_results_df.query('tumour_id == @tumour_id')) for clone in clones_in_other_events], [])
                forbidden_ancestors = sum([get_ancestors(clone, alpaca_wgd_results_df.query('tumour_id == @tumour_id')) for clone in clones_in_other_events], [])
        clones_in_other_events = [c.split('_')[-1] if '_' in str(c) else str(c) for c in other_events.clusters.astype(str).tolist()]
        forbidden_clones = forbidden_descendants + clones_in_other_events + forbidden_ancestors
        clone_proportions_in_selected_tumour = load_clone_proportions_for_tumour(tumour_id, alpaca_input_dir)

        # we only load wgd_results for convenience, because this table already contains phylogeny:
        tree_df = alpaca_wgd_results_df.query('tumour_id == @tumour_id')[['clone', 'parent']]
        # is the event found only in one sample?
        single_sample_event = event_subdf.samples.str.contains(',').sum() == 0
        if single_sample_event:
            # We are not using ALPACA ratios in these cases, as they are wrong. MRCA is usually absent and ALPACA solves it parsimoniously, postulating clonal GDs.
            samples_from_pgdd_event = event_subdf.samples.iloc[0].split(',')
            cp_samples = clone_proportions_in_selected_tumour[samples_from_pgdd_event]
            clones_present = cp_samples.sum(axis=1)
            clones_present = clones_present[clones_present > 0].index.tolist()
            # add all the extinct ancestors to the search space:
            all_ancestors = sum([get_ancestors(clone, tree_df) for clone in clones_present],[])
            clones_present_with_ancestors = list(set(clones_present + all_ancestors))
            # remove diploid:
            clones_present_with_ancestors = [clone for clone in clones_present_with_ancestors if clone != 'diploid']
            clones_present_with_ancestors_ordered = order_clones_by_phylogeny(clones_present_with_ancestors, tree_df)
            # starting from the terminal node, find the last clone where GD can be placed:
            origin_candidates = {x: [] for x in range(len(clones_present_with_ancestors_ordered))}
            lineage_index = 0
            for lineage in clones_present_with_ancestors_ordered:
                logger.log("TEST", f"Checking lineage {lineage}")
                for clone in lineage:
                    logger.log("TEST", f"Checking clone {clone}")
                    # if the clone is terminal, it can be the origin:
                    if clone == lineage[0] and not clone in forbidden_clones:
                        logger.log("TEST", f"Clone is terminal, appending")
                        origin_candidates[lineage_index].append(clone)
                        continue
                    # if the clone is not terminal,  check if all its children are present in the sample:
                    all_descendants = get_descendants(clone, tree_df)
                    # check if any of the descendants is in the forbidden_clones list. It contains clones implicated in other GD events of this tumour.
                    # TODO this requires checking if the GD events are of the same order to allow sequential GDs (i.e. ploidy supported event following a previous mutation supported event.)
                    if any(descendant in forbidden_clones for descendant in all_descendants):
                        continue
                    if clone in forbidden_clones:
                        continue
                    origin_candidates[lineage_index].append(clone)
                    logger.log("TEST", f"Clone is not terminal and not forbidden, appending")
                lineage_index += 1
            
            if origin_candidates:
                origin_candidates = [c for c in origin_candidates.values() if c]
                # if resulting list is empty, flag as potential PGDD issue:
                if len(origin_candidates) == 0:
                    events_df_resolved.loc[events_df_resolved.GD_event_id == event_id, 'clusters'] = 'false_call'
                    continue
                # if there is only one non-empty list, pick the last clone from it:
                if len(origin_candidates) == 1:
                    origin_candidates = origin_candidates[0]
                    origin_clone = origin_candidates[-1]
                else:
                    # if all lists terminate with the same node, pick it:
                    if all(c[-1] == origin_candidates[0][-1] for c in origin_candidates):
                        origin_clone = origin_candidates[0][-1]
                    else:
                        logger.log("TEST", f"Multiple origin candidates found for event {event_id} in tumour {tumour_id}: {origin_candidates}. Cannot resolve placement without further information.")
                        continue
                print(f"Event {event_id} is placed in clone {origin_clone} in tumour {tumour_id}.")
                # write back to events_df:
                events_df_resolved.loc[events_df_resolved.GD_event_id == event_id, 'clusters'] = 'ploidy_' + origin_clone
        else:
            print(f"Event {event_id} is found in multiple samples: {event_subdf.samples.iloc[0]}")
            continue


In [ ]:
# QC: for each tumour, sum the number of GD events on each tree path (mrc-leaf lineage). Then compare this number to per-sample ploidy.
# test inspired by LTX1167-Tumour1, where PGDD called mutation based and ploidy based event, but the latter is erroneous because clone2 GD already explains the ploidy of samples.
all_tumours = events_df_resolved.tumour_id.unique()
test_tumours = []#'LTX1167-Tumour1']
if test_tumours:
    all_tumours = test_tumours
mismatch_found = []
for tumour_id in all_tumours:
    #logger.log('PROD', f"Processing tumour {tumour_id}...")
    events_subdf = events_df_resolved[events_df_resolved.tumour_id == tumour_id]
    clonal_only = (events_subdf.is_clonal == True).all()
    if clonal_only:
        logger.log('TEST', f"Tumour {tumour_id} has only clonal GD events, skipping QC.")
        continue    
    tree_df = alpaca_wgd_results_df.query('tumour_id == @tumour_id')[['clone', 'parent']]
    terminal_clones = tree_df[~tree_df.clone.isin(tree_df.parent)].clone.tolist()
    leaf_gd_counts = {}
    gd_clusters = events_subdf.clusters.unique().tolist()
    if pd.isna(gd_clusters).any():
        logger.log('TEST', f"Tumour {tumour_id} has ploidy based GD events, skipping QC.")
        continue
    if 'false_call' in gd_clusters:
        logger.log('TEST', f"Tumour {tumour_id} has false call GD events, skipping QC.")
        continue
    gd_clusters_numeric = [int(c.split('clone')[1]) if '_' in c else int(c) for c in gd_clusters ]
    gd_clones = [f'clone{c}' for c in gd_clusters_numeric]
    for terminal_clone in terminal_clones:
        # get the path from terminal clone to root:
        path = [terminal_clone]
        while path[-1] != 'diploid':
            parent = tree_df[tree_df.clone == path[-1]].parent.iloc[0]
            path.append(parent)
        # count the number of GD events on this path:
        gd_count = len([c for c in path if c in gd_clones])
        leaf_gd_counts[terminal_clone] = gd_count
    max_gd_level = summary_df[summary_df.tumour_id == tumour_id].num_gds.max()
    max_gd_level_from_tree = max(leaf_gd_counts.values())
    if max_gd_level != max_gd_level_from_tree:
        mismatch_found.append(tumour_id)
        logger.log('TEST', f"QC failed for tumour {tumour_id}: max GD level from summary ({max_gd_level}) does not match max GD level from tree ({max_gd_level_from_tree}).")
if mismatch_found:
    print(f"QC failed for the following tumours: {mismatch_found}")

In [ ]:
m_iter = iter(mismatch_found)

In [ ]:
# starting with 18 tumours and 1 hr


t = next(m_iter)
print(t)
idx=mismatch_found.index(t)
all_items = len(mismatch_found)
print(f"Processing tumour {idx+1}/{all_items}: {t}")
which_q(t)


In [ ]:
events_df_resolved[events_df_resolved.tumour_id == t]

In [ ]:
# identify all the cases with reversion to diploid.

# TODO replace this with a mutation in the final run:
events_df_resolved_resolved=events_df_resolved.copy()

alpaca_results_df = get_alpaca_results(alpaca_output_dir)
estimated_ploidy = alpaca_results_df.groupby(['tumour_id','clone']).agg({'pred_CN_A':'median','pred_CN_B':'median'}).reset_index()

all_tumours = events_df_resolved.tumour_id.unique()
test_tumours = []
if test_tumours:
    all_tumours = [tumour_id for tumour_id in all_tumours if tumour_id in test_tumours]
reversion_found = []
for tumour_id in all_tumours:
    if tumour_id not in gd_to_process_mut_support.tumour_id.unique():
        continue
    events_subdf = events_df_resolved[events_df_resolved.tumour_id == tumour_id]
    # ignore events which were not assigned to clusters:
    events_subdf = events_subdf[~events_subdf.clusters.isna()]
    for event in events_subdf.itertuples():
        if event.is_clonal:
            continue
        cluster = event.clusters
        if '_' in cluster:
            clone = cluster.split('_')[1]
        else:
            clone = f'clone{cluster}'
        # get descendants of this clone:
        descendants = get_descendants(clone, alpaca_wgd_results_df.query('tumour_id == @tumour_id'))
        descendants_ploidy = estimated_ploidy[(estimated_ploidy.tumour_id==tumour_id) & (estimated_ploidy.clone.isin(descendants))]
        if (descendants_ploidy['pred_CN_A'] == 1).any():
            logger.log('TEST', f"Reversion to diploid found in tumour {tumour_id} for event {event.GD_event_id} in clone {clone}.")
            reversion_found.append(tumour_id)
            events_df_resolved_resolved.loc[events_df_resolved_resolved.GD_event_id == event.GD_event_id, 'clusters'] = 'reversion_'+events_df_resolved_resolved.loc[events_df_resolved_resolved.GD_event_id == event.GD_event_id, 'clusters']
        

In [ ]:
reversion_iter = iter(reversion_found)

In [ ]:
t=next(reversion_iter, None)
print(t)
which_q(t)

In [ ]:
selected_tumour_id='LTX1315-Tumour1'
which_q(selected_tumour_id)

In [ ]:
# check for any evidence of reversion in ploidy only calls:
lowest_mrca_child_ratio = round(alpaca_wgd_results_df[alpaca_wgd_results_df['parent']==mrca]['ratio_A'].min(),2)
lowest_mrca_child_ratio = lowest_mrca_child_ratio
ALPACA_clonal_wgd_ratio = alpaca_wgd_results_df[alpaca_wgd_results_df['parent']=='diploid']['ratio_A'].values[0].round(2)

# generate final table
Make the table with calls assigned to clones. This will be ingested by the viewer. All the subclonal GDs will be qc-ed manually.

In [ ]:
template = get_alpaca_results(alpaca_output_dir)[['tumour_id', 'clone']].drop_duplicates()
qc_df, qc_df_finalised_tumours = get_qc_df(QC_table_path)
# additional QC: remove tumours where CCF values are not ~100 in the clonal cluster in all regions:
df_all_trunk, extra_qc_df_finalised_tumours = additional_ccf_qc(qc_df_finalised_tumours,ccf_thr=0.9, release_path='/nemo/project/proj-tracerx-lung/tracerx/_RELEASE/release_tx842')

In [ ]:
df_all_trunk.meanCCF.hist()

In [ ]:
template['GD'] = 0
template['tx_QC'] = template['tumour_id'].isin(qc_df_finalised_tumours['patient_tumour'])
template['ccf_QC'] = template['tumour_id'].isin(extra_qc_df_finalised_tumours['patient_tumour'])
template['QC_pass'] = template['tx_QC'] & template['ccf_QC']
template['Quality'] = np.nan

In [ ]:
# add clonal only cases
MRCA_clones = alpaca_wgd_results_df[alpaca_wgd_results_df.parent=='diploid'][['tumour_id','clone']]
count_before = len(per_tumour_df_clonal_gd_only_queue)
clonal_only = per_tumour_df_clonal_gd_only_queue.merge(MRCA_clones, how='left')
count_after = len(clonal_only)
assert count_before == count_after, f"Counts before and after merge do not match: {count_before} vs {count_after}"
# find the GD count:
clonal_gd_count = per_tumour_df[per_tumour_df.tumour_id.isin(clonal_only.tumour_id)][['tumour_id', 'num_clonal_gds']]
clonal_only = clonal_only.merge(clonal_gd_count, how='left')
for tumour_id in clonal_only.tumour_id.unique():
    template.loc[template.tumour_id == tumour_id, 'GD'] = 0
    template.loc[template.tumour_id == tumour_id, 'Quality'] = 1
    GD_clone = clonal_only[clonal_only.tumour_id == tumour_id]['clone'].values[0]
    GD_count = clonal_only[clonal_only.tumour_id == tumour_id]['num_clonal_gds'].values[0]
    template.loc[(template.tumour_id == tumour_id)&(template.clone == GD_clone), 'GD'] = GD_count

In [ ]:
# add no GD cases:
per_tumour_df_no_gd_queue
for tumour_id in per_tumour_df_no_gd_queue.tumour_id.unique():
    template.loc[template.tumour_id == tumour_id, 'GD'] = 0
    template.loc[template.tumour_id == tumour_id, 'Quality'] = 1

In [ ]:
# export the pipeline summary and in the first pass deal with all the tumours manually

In [ ]:
#template.to_csv(f'{queue_files}/june_reconciliation.tsv', sep='\t', index=False)

Add supporting mutation count to events df. We do this here, because we need to resolve the multi-cluster support first. 

In [ ]:
events_df_exploded_mut_count.query('tumour_id == @event.tumour_id')

In [ ]:
events_df['supporting_mut_count'] = np.nan
for event in events_df.itertuples():
    is_mut_supported = event.is_subclonal_mutation_supported
    if is_mut_supported:
        cluster = event.clusters
        if '_' in cluster:
            cluster = cluster.split('_clone')[1]
        cluster_numeric = int(cluster)
        count_df = mut_count_df.query('tumour_id == @event.tumour_id').query('cluster_id == @cluster_numeric')
        count_df = count_df[(count_df.discover_pass_all) | (count_df.check_pass_all)]
        count_df = count_df.sort_values('num_cn2', ascending=False)
        supporting_count = count_df.num_cn2.iloc[0]
        events_df.at[event.Index, 'supporting_mut_count'] = supporting_count

events_df_backup_path = events_path.replace('.tsv', '_backup.tsv')
if not os.path.exists(events_df_backup_path):
    events_df.to_csv(events_df_backup_path, sep='\t', index=False)
if overwrite or not os.path.exists(events_path):
    events_df.to_csv(events_path.replace('.tsv','_with_counts.tsv'), sep='\t', index=False)

In [ ]:
events_path

In [ ]:
# now, load the results of the 'generate output' button from the viewer. Highlight tumours of low quality. Then add the mutation count context to further improve the quality calls and calculate the Bielski fraction for the same purpose.
# Note: the output must be copied from local to nemo first:
# cp /nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/wgd_viewer/data/output/clone_gd_calls_latest.tsv /nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/wes/release_tx842/latest_viewer_output

In [ ]:
import pandas as pd
viewer_output_path = '/nemo/project/proj-tracerx-lung/tctProjects/CN-CCF/pgdd/wes/release_tx842/latest_viewer_output/clone_gd_calls_latest.tsv'
vdf = pd.read_table(viewer_output_path)

In [ ]:
events_df['clone_id'] = events_df['clusters'].apply(lambda x: x.split('_')[-1] if '_' in str(x) else str(x))
events_df['clone_id'] = events_df['clone_id'].apply(lambda x: f'clone{x}' if 'clone' not in str(x) else str(x))

In [ ]:
vdf_counts = vdf.merge(events_df[['tumour_id','clone_id','supporting_mut_count']], how='left', left_on=['tumour_id','clone_id'], right_on=['tumour_id','clone_id'])
vdf_counts_with_mutations = vdf_counts.dropna(subset=['supporting_mut_count'])
vdf_counts_with_mutations.supporting_mut_count.hist()

In [ ]:
counts_clipped_at_25 = vdf_counts.supporting_mut_count.dropna().clip(upper=25)
counts_clipped_at_25.hist()

In [ ]:
vdf[vdf.quality <1]

In [ ]:
low_mut_count = vdf_counts_with_mutations[vdf_counts_with_mutations.supporting_mut_count <= vdf_counts_with_mutations.supporting_mut_count.quantile(0.05)]

In [ ]:
vdf.loc[vdf.tumour_id.isin(low_mut_count.tumour_id), 'quality'] = 0.5

In [ ]:
vdf['quality'] = vdf['quality'].fillna(1)
vdf['quality'] = vdf.groupby('tumour_id')['quality'].transform('min')

In [ ]:

vdf['quality'] = vdf['quality'].apply(lambda x: 'Low' if x < 1 else 'High')
vdf = vdf[['tumour_id', 'clone_id', 'number_of_gds', 'quality', 'status', 'queue']]
vdf = vdf.rename(columns={'queue': 'class'})
vdf['total_gds'] = vdf.groupby('tumour_id')['number_of_gds'].transform('sum')
tree_structure = alpaca_wgd_results_df[['tumour_id', 'clone', 'parent']]
vdf = vdf.merge(tree_structure, how='left', left_on=['tumour_id', 'clone_id'], right_on=['tumour_id', 'clone'])
#vdf.to_csv(f'{queue_files}/latest_viewer_output_with_mutation_counts_and_classification.tsv', sep='\t', index=False)